In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
# Helper functions
from anthropic.types import Message
# Magic string to trigger redacted thinking
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    #확장 사고 기능 추가
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
import json, re

def grade_by_model(question, output):
    eval_prompt = f"""당신은 K-IFRS에 정통한 숙련된 회계사입니다. 아래 AI의 현금흐름표 분석 답변을 평가해주세요.

원본 질문:
<question>
{question}
</question>

평가 대상 답변:
<answer>
{output}
</answer>

평가 기준:
1. 각 분개의 현금 수반 여부 판단 정확성
2. 현금 거래의 영업/투자/재무활동 분류 정확성
3. 현금흐름표 양식 내 금액 배치 정확성
4. 설명의 명확성과 논리적 일관성

채점 원칙:
- K-IFRS IAS 7을 기준으로 엄격히 평가하세요
- 현금흐름 분류는 계정과목 이름이 아니라 해당 현금흐름이 발생한 거래의 경제적 실질을 기준으로 판단하세요
- 특히 부채 관련 현금흐름은 그 부채가 어떤 목적으로 발생했는지를 반드시 추적하여 분류 적정성을 검토하세요

마크다운 없이 순수 JSON만 출력하세요:
{{
    "strengths": ["..."],
    "weaknesses": ["..."],
    "reasoning": "...",
    "score": 숫자
}}"""
    grading_messages = [{"role": "user", "content": eval_prompt}]
    raw = text_from_message(chat(grading_messages))
    json_str = re.search(r'\{.*\}', raw, re.DOTALL).group()
    return json.loads(json_str)

In [4]:
messages = [
    {
        "role": "user",
        "content": """다음은 A회사의 2025년 유형자산 관련 간이 분개장입니다.

<journal_entries>
[1] 03/15 차) 기계장치    800  /  대) 보통예금     800
    - 생산라인 증설용 기계장치 취득

[2] 06/01 차) 차량운반구   200  /  대) 미지급금     200
    - 영업용 차량 리스 아닌 외상 취득

[3] 09/30 차) 보통예금     150  /  대) 비품        300
              유형자산처분손실 150
    - 노후 사무용 비품 처분 (취득원가 500, 감가상각누계액 200)

[4] 12/31 차) 감가상각비   450  /  대) 감가상각누계액  450
    - 연간 감가상각 (기계장치 200, 차량운반구 100, 비품 150)

[5] 12/31 차) 미지급금     100  /  대) 보통예금     100
    - 차량운반구 외상대금 중 일부 상환
</journal_entries>

<instructions>
위 분개장을 분석하여:
1. 각 분개의 현금 수반 여부를 판단하시오.
2. 현금이 수반된 거래를 영업활동/투자활동/재무활동으로 분류하시오.
3. 아래 현금흐름표 양식의 해당 항목에 금액을 배치하시오.
</instructions>

<cash_flow_template>
[현금흐름표 - 간접법]

I. 영업활동 현금흐름
   당기순이익: (별도 제공)
   조정항목:
     감가상각비: ___
     유형자산처분손실: ___

II. 투자활동 현금흐름
     유형자산 취득: ___
     유형자산 처분: ___

III. 재무활동 현금흐름
     ___
</cash_flow_template>
"""
    }
]

# --- thinking 없이 실행 ---
question = messages[0]["content"]
msgs_plain = [{"role": "user", "content": question}]
output_plain = text_from_message(chat(msgs_plain))                          # 텍스트 반환

# --- thinking 켜고 실행 ---
msgs_think = [{"role": "user", "content": question}]
response_think = chat(msgs_think, thinking=True, thinking_budget=4096)
output_think = text_from_message(response_think)        # Message에서 텍스트 추출

# --- 각각 채점 ---
grade_plain = grade_by_model(question, output_plain)
grade_think = grade_by_model(question, output_think)

# --- 결과 비교 ---
print("=== Thinking 미사용 ===")
print(output_plain)
print(f"점수: {grade_plain['score']}/10")
print(f"평가: {grade_plain['reasoning']}")

print("\n=== Extended Thinking 사용 ===")
print(output_think)
print(f"점수: {grade_think['score']}/10")
print(f"평가: {grade_think['reasoning']}")


=== Thinking 미사용 ===
# 현금흐름표 분석

## 1단계: 각 분개의 현금 수반 여부 판단

| 분개 | 거래 내용 | 현금 계정 | 현금 수반 여부 |
|------|-----------|-----------|----------------|
| [1] | 기계장치 취득 | 보통예금 800 지출 | ✅ **현금 유출** (-800) |
| [2] | 차량운반구 외상 취득 | 미지급금 계상 | ❌ **비현금** (외상) |
| [3] | 비품 처분 | 보통예금 150 수취 | ✅ **현금 유입** (+150) |
| [4] | 감가상각비 계상 | 현금 없음 | ❌ **비현금** (장부조정) |
| [5] | 미지급금 상환 | 보통예금 100 지출 | ✅ **현금 유출** (-100) |

---

## 2단계: 현금 거래의 활동별 분류

| 분개 | 현금흐름 | 분류 | 근거 |
|------|----------|------|------|
| [1] 기계장치 취득 | -800 | **투자활동** | 유형자산 취득은 설비투자 |
| [3] 비품 처분 | +150 | **투자활동** | 유형자산 매각 수입 |
| [5] 미지급금 상환 | -100 | **재무활동** | 부채(미지급금) 상환 |

> **⚠️ 분개[2] 추가 설명**
> 차량운반구 외상 취득은 현금 거래가 아니므로 현금흐름표 본문에 직접 반영되지 않습니다.
> 다만, **주석에 비현금 투자·재무활동으로 별도 공시**해야 합니다.

---

## 3단계: 현금흐름표 작성

```
════════════════════════════════════════════
         현금흐름표 (간접법)
════════════════════════════════════════════

I. 영업활동 현금흐름
   당기순이익                        XXX  ← 별도 제공
   
   [조정항목] ─ 비현금 항목 가산
     감가상각비                    + 450 

In [ ]:
# 확장사고 미사용시에는 유형자산에 대한 미지급금상환을 재무로 잘못 분류하였고, 평가 프롬프트가 이를 잘 짚어냄.
# 다만 확장사고 사용시에 평가 함수가 위와는 다른 진술을 하며 미지급금상환을 재무로 분류하는게 맞다는 오류 발생.
# 현금흐름표 작성시는 확장사고 사용이 보다 정확도 높은 답변을 도출, 평가 프롬프트는 수정이 필요한 것으로 보이나, 본 예제의 목적은 아님
